# Chess.com Performance Analysis

### Project Overview
This project analyzes **3,257 rated games** played on Chess.com from March 2023
to March 2026, using data retrieved directly via the Chess.com Public API.

The objective is to examine rating progression, opening repertoire effectiveness,
and performance trends across Bullet, Blitz, and Rapid time controls, uncovering
measurable patterns that translate into actionable improvement recommendations.

Rather than reviewing individual games, this analysis focuses on identifying
systematic strengths, weaknesses, and performance drivers across all three formats.

This notebook is built to be fully reproducible. Any Chess.com user can substitute
their own username to replicate the entire analysis pipeline.

---

### Target Ratings
| Time Control | Current Rating | Target Rating |
|---|---|---|
| Rapid | 1475 | 2000 |
| Blitz | 900 | 1500 |
| Bullet | 750 | 1500 |

---

### Key Questions

1. Which openings (as White and as Black) produce the highest and lowest win rates,
   and what does this reveal about where to focus study and preparation?

2. Which openings are associated with the highest accuracy scores, and do high
   accuracy games correlate with better results?

3. How has my rating progressed over time across each time control, and are there
   identifiable periods of improvement or regression?

4. Does my performance differ meaningfully when playing as White versus Black,
   and which color exposes more weaknesses?

5. How do win/loss/draw rates vary across Bullet, Blitz, and Rapid, and what
   does this suggest about where time pressure affects my play the most?

## Importing and understanding general overview of data

In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [2]:
raw_chess_data = pd.read_csv("../data/chess_games_raw.csv")
print(f"Raw dataset shape: {raw_chess_data.shape}")
print(f"Total games loaded: {len(raw_chess_data)}")

Raw dataset shape: (3257, 26)
Total games loaded: 3257


In [3]:
raw_chess_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 3257 entries, 0 to 3256
Data columns (total 26 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   url               3257 non-null   str    
 1   pgn               3257 non-null   str    
 2   time_control      3257 non-null   str    
 3   end_time          3257 non-null   int64  
 4   rated             3257 non-null   bool   
 5   tcn               3254 non-null   str    
 6   uuid              3257 non-null   str    
 7   initial_setup     3257 non-null   str    
 8   fen               3257 non-null   str    
 9   time_class        3257 non-null   str    
 10  rules             3257 non-null   str    
 11  eco               3250 non-null   str    
 12  accuracies.white  653 non-null    float64
 13  accuracies.black  653 non-null    float64
 14  white.rating      3257 non-null   int64  
 15  white.result      3257 non-null   str    
 16  white.@id         3257 non-null   str    
 17  white.

In [4]:
raw_chess_data.describe()

,end_time,accuracies.white,accuracies.black,white.rating,black.rating,start_time
count,3.257000e+03,653.000000,653.000000,3257.000000,3257.000000,3.000000e+00
mean,1.738760e+09,71.656799,70.717305,902.826527,903.440282,1.764309e+09
std,2.409356e+07,12.089701,12.014223,208.798512,207.922279,9.658591e+04
min,1.678545e+09,17.240000,9.710000,0.000000,0.000000,1.764253e+09
25%,1.726675e+09,64.290000,63.590000,796.000000,796.000000,1.764253e+09
50%,1.737819e+09,72.440000,71.630000,876.000000,875.000000,1.764253e+09
75%,1.761409e+09,80.120000,78.760000,987.000000,991.000000,1.764337e+09
max,1.773953e+09,100.000000,100.000000,1906.000000,2202.000000,1.764420e+09


In [5]:
raw_chess_data.columns

Index(['url', 'pgn', 'time_control', 'end_time', 'rated', 'tcn', 'uuid',
       'initial_setup', 'fen', 'time_class', 'rules', 'eco',
       'accuracies.white', 'accuracies.black', 'white.rating', 'white.result',
       'white.@id', 'white.username', 'white.uuid', 'black.rating',
       'black.result', 'black.@id', 'black.username', 'black.uuid',
       'tournament', 'start_time'],
      dtype='str')

## Data Cleaning

In [6]:
# Null value check
raw_chess_data.isnull().sum()

url                    0
pgn                    0
time_control           0
end_time               0
rated                  0
tcn                    3
uuid                   0
initial_setup          0
fen                    0
time_class             0
rules                  0
eco                    7
accuracies.white    2604
accuracies.black    2604
white.rating           0
white.result           0
white.@id              0
white.username         0
white.uuid             0
black.rating           0
black.result           0
black.@id              0
black.username         0
black.uuid             0
tournament          3076
start_time          3254
dtype: int64

In [7]:
# Drop columns not needed for analysis
cols_to_drop = [
    'tcn',           # encoded move notation — not needed
    'initial_setup', # always standard chess starting position
    'fen',           # final board position — not needed
    'tournament',    # 94% null, out of scope
    'start_time',    # 99.9% null
    'white.@id',     # API URL — redundant with username
    'black.@id',     # API URL — redundant with username
    'white.uuid',    # internal ID — not needed
    'black.uuid',    # internal ID — not needed
    'uuid',          # internal ID — not needed
    'rules',         # all values are 'chess' — no variance
]

raw_chess_data.drop(columns=cols_to_drop, inplace=True)
print(f"Columns after dropping irrelevant fields: {raw_chess_data.shape[1]}")

Columns after dropping irrelevant fields: 15


In [8]:
# Remove daily (correspondence) games — out of scope for this analysis
raw_chess_data = raw_chess_data[raw_chess_data['time_class'] != 'daily']
print(f"Games after removing daily games: {len(raw_chess_data)}")

Games after removing daily games: 3254


In [9]:
# Your Chess.com username (change this to make the notebook reproducible)
USERNAME = 'Fareed04'

# Determine which color you played
raw_chess_data['player_color'] = raw_chess_data['white.username'].apply(
    lambda x: 'white' if x.lower() == USERNAME.lower() else 'black'
)

# Derive your rating and opponent rating per game
raw_chess_data['player_rating'] = raw_chess_data.apply(
    lambda row: row['white.rating'] if row['player_color'] == 'white'
    else row['black.rating'], axis=1
)

raw_chess_data['opponent_rating'] = raw_chess_data.apply(
    lambda row: row['black.rating'] if row['player_color'] == 'white'
    else row['white.rating'], axis=1
)

# Derive opponent username
raw_chess_data['opponent_username'] = raw_chess_data.apply(
    lambda row: row['black.username'] if row['player_color'] == 'white'
    else row['white.username'], axis=1
)

print(raw_chess_data['player_color'].value_counts())

player_color
black    1633
white    1621
Name: count, dtype: int64


In [10]:
# Map all result types into Win / Draw / Loss from the player's perspective
win_results = {'win'}
draw_results = {'repetition', 'stalemate', 'agreed', 'insufficient', 'timevsinsufficient'}
loss_results = {'checkmated', 'resigned', 'timeout', 'abandoned'}

def get_player_result(row):
    if row['player_color'] == 'white':
        result = row['white.result']
    else:
        result = row['black.result']
    
    if result in win_results:
        return 'Win'
    elif result in draw_results:
        return 'Draw'
    elif result in loss_results:
        return 'Loss'
    else:
        return 'Unknown'

raw_chess_data['result'] = raw_chess_data.apply(get_player_result, axis=1)

print(raw_chess_data['result'].value_counts())

result
Win     1694
Loss    1451
Draw     109
Name: count, dtype: int64


In [11]:
# ECO column contains full URLs e.g.:
# https://www.chess.com/openings/Nimzowitsch-Larsen-Attack-Modern-Variation
# Extracting just the opening name and clean it up

def parse_opening(eco_url):
    if pd.isna(eco_url):
        return 'Unknown'
    # Extract the slug after /openings/
    match = re.search(r'/openings/([^/]+)$', eco_url)
    if match:
        slug = match.group(1)
        # Replace hyphens with spaces
        return slug.replace('-', ' ')
    return 'Unknown'

raw_chess_data['opening'] = raw_chess_data['eco'].apply(parse_opening)

# Also extract the base opening family (first 2 words)
raw_chess_data['opening_family'] = raw_chess_data['opening'].apply(
    lambda x: ' '.join(x.split()[:2]) if x != 'Unknown' else 'Unknown'
)

print(f"Unique openings: {raw_chess_data['opening'].nunique()}")
print(f"\nTop 10 most played openings:")
print(raw_chess_data['opening'].value_counts().head(10))

Unique openings: 456

Top 10 most played openings:
opening
Queens Pawn Opening Accelerated London System                    207
Scandinavian Defense Mieses Kotrc Variation 3.Nc3 Qd8 4.Nf3      164
Scandinavian Defense                                             160
Scandinavian Defense 2.e5                                        143
Scandinavian Defense Mieses Kotrc Variation 3.Nc3 Qd8             68
Englund Gambit 2.dxe5                                             68
Queens Pawn Opening Accelerated London System 2...Nf6 3.e3        60
Scandinavian Defense Mieses Kotrc Variation 3.Nc3 Qd8 4.d4        55
Queens Pawn Opening Horwitz Defense 2.Bf4                         47
Queens Pawn Opening Accelerated London System 2...Bf5 3.e3 e6     45
Name: count, dtype: int64


In [12]:
# Converting Unix timestamp to datetime
raw_chess_data['date'] = pd.to_datetime(raw_chess_data['end_time'], unit='s')
raw_chess_data['year'] = raw_chess_data['date'].dt.year
raw_chess_data['month'] = raw_chess_data['date'].dt.month
raw_chess_data['month_year'] = raw_chess_data['date'].dt.to_period('M')
raw_chess_data['day_of_week'] = raw_chess_data['date'].dt.day_name()

print(f"Date range: {raw_chess_data['date'].min()} to {raw_chess_data['date'].max()}")
print(f"\nGames per year:")
print(raw_chess_data['year'].value_counts().sort_index())

Date range: 2023-03-11 14:21:58 to 2026-03-19 20:42:37

Games per year:
year
2023     317
2024    1070
2025    1459
2026     408
Name: count, dtype: int64


In [13]:
# Assigning accuracy per game based on color played
raw_chess_data['player_accuracy'] = raw_chess_data.apply(
    lambda row: row['accuracies.white'] if row['player_color'] == 'white'
    else row['accuracies.black'], axis=1
)

raw_chess_data['opponent_accuracy'] = raw_chess_data.apply(
    lambda row: row['accuracies.black'] if row['player_color'] == 'white'
    else row['accuracies.white'], axis=1
)

print(f"Games with accuracy data: {raw_chess_data['player_accuracy'].notna().sum()}")
print(f"Games without accuracy data: {raw_chess_data['player_accuracy'].isna().sum()}")
print(f"\nAverage accuracy where available: {raw_chess_data['player_accuracy'].mean():.2f}%")

Games with accuracy data: 653
Games without accuracy data: 2601

Average accuracy where available: 72.84%


In [14]:
# Dropping raw columns now replaced by cleaner derived ones
raw_chess_data.drop(columns=[
    'white.result', 'black.result',
    'white.username', 'black.username',
    'white.rating', 'black.rating',
    'accuracies.white', 'accuracies.black',
    'eco', 'end_time', 'url'
], inplace=True)

# Reordering columns logically
raw_chess_data = raw_chess_data[[
    'date', 'year', 'month', 'month_year', 'day_of_week',
    'time_class', 'time_control',
    'player_color', 'player_rating', 'opponent_rating', 'opponent_username',
    'result', 'opening', 'opening_family',
    'player_accuracy', 'opponent_accuracy',
    'rated', 'pgn'
]]

# Final shape check
print(f"Cleaned dataset shape: {raw_chess_data.shape}")
print(f"\nNull check:")
print(raw_chess_data.isnull().sum())

# Exporting cleaned dataset
raw_chess_data.to_csv('../data/chess_games_cleaned.csv', index=False)
print("\nchess_games_cleaned.csv saved successfully")

Cleaned dataset shape: (3254, 18)

Null check:
date                    0
year                    0
month                   0
month_year              0
day_of_week             0
time_class              0
time_control            0
player_color            0
player_rating           0
opponent_rating         0
opponent_username       0
result                  0
opening                 0
opening_family          0
player_accuracy      2601
opponent_accuracy    2601
rated                   0
pgn                     0
dtype: int64

chess_games_cleaned.csv saved successfully


### Analysis

In [17]:
# Load cleaned dataset
cleaned_chess_data = pd.read_csv('../data/chess_games_cleaned.csv')

# Restore month_year as Period (lost during CSV export)
cleaned_chess_data['month_year'] = pd.to_datetime(cleaned_chess_data['date']).dt.to_period('M')
cleaned_chess_data['date'] = pd.to_datetime(cleaned_chess_data['date'])

print(f"Cleaned dataset loaded: {cleaned_chess_data.shape}")
print(f"\nTime controls:")
print(cleaned_chess_data['time_class'].value_counts())

Cleaned dataset loaded: (3254, 18)

Time controls:
time_class
blitz     1722
rapid     1430
bullet     102
Name: count, dtype: int64


In [18]:
# SECTION 1 — OVERALL PERFORMANCE SUMMARY

# Overall win/loss/draw distribution
overall = cleaned_chess_data['result'].value_counts().reset_index()
overall.columns = ['Result', 'Games']
overall['Percentage'] = (overall['Games'] / len(cleaned_chess_data) * 100).round(1)

print("=== Overall Results ===")
print(overall.to_string(index=False))

# Results by time control
print("\n=== Results by Time Control ===")
by_time = cleaned_chess_data.groupby(['time_class', 'result']).size().unstack(fill_value=0)
by_time['Total'] = by_time.sum(axis=1)
by_time['Win %'] = (by_time['Win'] / by_time['Total'] * 100).round(1)
by_time['Loss %'] = (by_time['Loss'] / by_time['Total'] * 100).round(1)
by_time['Draw %'] = (by_time['Draw'] / by_time['Total'] * 100).round(1)
print(by_time.to_string())

=== Overall Results ===
Result  Games  Percentage
   Win   1694        52.1
  Loss   1451        44.6
  Draw    109         3.3

=== Results by Time Control ===
result      Draw  Loss  Win  Total  Win %  Loss %  Draw %
time_class                                               
blitz         49   792  881   1722   51.2    46.0     2.8
bullet         2    36   64    102   62.7    35.3     2.0
rapid         58   623  749   1430   52.4    43.6     4.1


In [19]:
# SECTION 2 — PERFORMANCE BY COLOR 

# Overall performance by color
print("=== Overall Results by Color ===")
by_color = cleaned_chess_data.groupby(['player_color', 'result']).size().unstack(fill_value=0)
by_color['Total'] = by_color.sum(axis=1)
by_color['Win %'] = (by_color['Win'] / by_color['Total'] * 100).round(1)
by_color['Loss %'] = (by_color['Loss'] / by_color['Total'] * 100).round(1)
by_color['Draw %'] = (by_color['Draw'] / by_color['Total'] * 100).round(1)
print(by_color.to_string())

# Performance by color and time control
print("\n=== Results by Color and Time Control ===")
by_color_time = cleaned_chess_data.groupby(
    ['time_class', 'player_color', 'result']
).size().unstack(fill_value=0)
by_color_time['Total'] = by_color_time.sum(axis=1)
by_color_time['Win %'] = (by_color_time['Win'] / by_color_time['Total'] * 100).round(1)
by_color_time['Loss %'] = (by_color_time['Loss'] / by_color_time['Total'] * 100).round(1)
print(by_color_time[['Win %', 'Loss %', 'Total']].to_string())

=== Overall Results by Color ===
result        Draw  Loss  Win  Total  Win %  Loss %  Draw %
player_color                                               
black           53   753  827   1633   50.6    46.1     3.2
white           56   698  867   1621   53.5    43.1     3.5

=== Results by Color and Time Control ===
result                   Win %  Loss %  Total
time_class player_color                      
blitz      black          49.8    47.3    855
           white          52.5    44.8    867
bullet     black          56.4    41.8     55
           white          70.2    27.7     47
rapid      black          51.2    45.1    723
           white          53.6    42.0    707


In [20]:
# SECTION 3 — RATING PROGRESSION

# Latest rating per month per time control
print("=== Rating Progression by Time Control ===")
rating_progression = cleaned_chess_data.groupby(
    ['time_class', 'month_year']
)['player_rating'].last().reset_index()
rating_progression.columns = ['Time Control', 'Month', 'Rating']

for tc in ['rapid', 'blitz', 'bullet']:
    subset = rating_progression[rating_progression['Time Control'] == tc]
    if len(subset) > 0:
        print(f"\n{tc.capitalize()}:")
        print(f"  Start rating : {subset.iloc[0]['Rating']}")
        print(f"  Peak rating  : {subset['Rating'].max()} ({subset.loc[subset['Rating'].idxmax(), 'Month']})")
        print(f"  Current rating: {subset.iloc[-1]['Rating']}")
        print(f"  Net change   : {subset.iloc[-1]['Rating'] - subset.iloc[0]['Rating']:+d}")

# Monthly average rating per time control
print("\n=== Monthly Average Rating (last 6 months) ===")
recent = cleaned_chess_data[cleaned_chess_data['date'] >= '2025-09-01']
recent_rating = recent.groupby(
    ['time_class', 'month_year']
)['player_rating'].mean().round(0).reset_index()
print(recent_rating.to_string(index=False))

=== Rating Progression by Time Control ===

Rapid:
  Start rating : 437
  Peak rating  : 1476 (2026-03)
  Current rating: 1476
  Net change   : +1039

Blitz:
  Start rating : 483
  Peak rating  : 979 (2026-03)
  Current rating: 979
  Net change   : +496

Bullet:
  Start rating : 697
  Peak rating  : 893 (2025-10)
  Current rating: 754
  Net change   : +57

=== Monthly Average Rating (last 6 months) ===
time_class month_year  player_rating
     blitz    2025-09          842.0
     blitz    2025-10          867.0
     blitz    2025-11          867.0
     blitz    2025-12          893.0
     blitz    2026-01          887.0
     blitz    2026-02          901.0
     blitz    2026-03          964.0
    bullet    2025-10          834.0
    bullet    2025-12          893.0
    bullet    2026-03          683.0
     rapid    2025-09         1213.0
     rapid    2025-12         1223.0
     rapid    2026-01         1271.0
     rapid    2026-02         1346.0
     rapid    2026-03         1414.0


In [24]:
# SECTION 4 — OPENING ANALYSIS

# Minimum games threshold to filter out rare openings
MIN_GAMES = 20

# Win rate by opening family overall
print("=== Top Opening Families by Games Played ===")
opening_stats = cleaned_chess_data.groupby('opening_family').agg(
    Games=('result', 'count'),
    Wins=('result', lambda x: (x == 'Win').sum()),
    Losses=('result', lambda x: (x == 'Loss').sum()),
    Draws=('result', lambda x: (x == 'Draw').sum()),
).reset_index()
opening_stats['Win %'] = (opening_stats['Wins'] / opening_stats['Games'] * 100).round(1)
opening_stats['Loss %'] = (opening_stats['Losses'] / opening_stats['Games'] * 100).round(1)
opening_stats = opening_stats[opening_stats['Games'] >= MIN_GAMES].sort_values('Games', ascending=False)
print(opening_stats.to_string(index=False))

# As White
white_openings = cleaned_chess_data[cleaned_chess_data['player_color'] == 'white'].groupby('opening_family').agg(
    Games=('result', 'count'),
    Wins=('result', lambda x: (x == 'Win').sum()),
    Losses=('result', lambda x: (x == 'Loss').sum()),
).reset_index()
white_openings['Win %'] = (white_openings['Wins'] / white_openings['Games'] * 100).round(1)
white_openings = white_openings[white_openings['Games'] >= MIN_GAMES]

print("\n=== Best Performing Opening Families as White (min 20 games) ===")
print(white_openings.sort_values('Win %', ascending=False).head(10).to_string(index=False))

print("\n=== Worst Performing Opening Families as White (min 20 games) ===")
print(white_openings.sort_values('Win %', ascending=True).head(10).to_string(index=False))

# As Black
black_openings = cleaned_chess_data[cleaned_chess_data['player_color'] == 'black'].groupby('opening_family').agg(
    Games=('result', 'count'),
    Wins=('result', lambda x: (x == 'Win').sum()),
    Losses=('result', lambda x: (x == 'Loss').sum()),
).reset_index()
black_openings['Win %'] = (black_openings['Wins'] / black_openings['Games'] * 100).round(1)
black_openings = black_openings[black_openings['Games'] >= MIN_GAMES]

print("\n=== Best Performing Opening Families as Black (min 20 games) ===")
print(black_openings.sort_values('Win %', ascending=False).head(10).to_string(index=False))

print("\n=== Worst Performing Opening Families as Black (min 20 games) ===")
print(black_openings.sort_values('Win %', ascending=True).head(10).to_string(index=False))

=== Top Opening Families by Games Played ===
      opening_family  Games  Wins  Losses  Draws  Win %  Loss %
         Queens Pawn   1349   738     560     51   54.7    41.5
Scandinavian Defense    853   416     407     30   48.8    47.7
         Indian Game    187    90      92      5   48.1    49.2
      Englund Gambit    148    82      62      4   55.4    41.9
       Queens Gambit     76    30      46      0   39.5    60.5
 Nimzowitsch Defense     71    36      35      0   50.7    49.3
    Kings Fianchetto     49    20      26      3   40.8    53.1
      Modern Defense     49    21      23      5   42.9    46.9
   Alekhines Defense     45    24      21      0   53.3    46.7
          Old Benoni     45    20      24      1   44.4    53.3
       London System     38    17      19      2   44.7    50.0
               Van t     31    21       9      1   67.7    29.0
        Reti Opening     31    14      16      1   45.2    51.6
     English Defense     29    15      13      1   51.7    

In [25]:
# SECTION 5 — ACCURACY ANALYSIS

# Note: Based on 653 games with accuracy data (20.1% of total dataset)

accuracy_df = cleaned_chess_data[cleaned_chess_data['player_accuracy'].notna()].copy()
print(f"Games with accuracy data: {len(accuracy_df)}")

# Average accuracy by result
print("\n=== Average Accuracy by Result ===")
acc_by_result = accuracy_df.groupby('result')['player_accuracy'].agg(
    Games='count',
    Avg_Accuracy='mean',
    Min_Accuracy='min',
    Max_Accuracy='max'
).round(2)
print(acc_by_result.to_string())

# Average accuracy by time control
print("\n=== Average Accuracy by Time Control ===")
acc_by_tc = accuracy_df.groupby('time_class')['player_accuracy'].agg(
    Games='count',
    Avg_Accuracy='mean'
).round(2)
print(acc_by_tc.to_string())

# Average accuracy by color
print("\n=== Average Accuracy by Color ===")
acc_by_color = accuracy_df.groupby('player_color')['player_accuracy'].agg(
    Games='count',
    Avg_Accuracy='mean'
).round(2)
print(acc_by_color.to_string())

# Accuracy by result and time control
print("\n=== Average Accuracy by Result and Time Control ===")
acc_by_result_tc = accuracy_df.groupby(
    ['time_class', 'result']
)['player_accuracy'].mean().round(2).unstack()
print(acc_by_result_tc.to_string())

# Top 10 openings by accuracy (min 10 games with accuracy data)
print("\n=== Top 10 Openings by Average Accuracy (min 10 games) ===")
acc_by_opening = accuracy_df.groupby('opening_family').agg(
    Games=('player_accuracy', 'count'),
    Avg_Accuracy=('player_accuracy', 'mean')
).round(2).reset_index()
acc_by_opening = acc_by_opening[acc_by_opening['Games'] >= 10]
print(acc_by_opening.sort_values('Avg_Accuracy', ascending=False).to_string(index=False))

Games with accuracy data: 653

=== Average Accuracy by Result ===
        Games  Avg_Accuracy  Min_Accuracy  Max_Accuracy
result                                                 
Draw       25         77.57         59.34         89.58
Loss      252         67.27         25.00        100.00
Win       376         76.25         32.99        100.00

=== Average Accuracy by Time Control ===
            Games  Avg_Accuracy
time_class                     
blitz         186         72.05
bullet         15         72.93
rapid         452         73.16

=== Average Accuracy by Color ===
              Games  Avg_Accuracy
player_color                     
black           329         71.69
white           324         74.00

=== Average Accuracy by Result and Time Control ===
result       Draw   Loss    Win
time_class                     
blitz       75.53  66.93  75.84
bullet      77.40  63.55  76.23
rapid       78.12  67.52  76.41

=== Top 10 Openings by Average Accuracy (min 10 games) ===
      op